# RISC-V Assembly Code Generation - Training 350M Model

Train a CodeGen-350M model on Simple compiler dataset for RISC-V assembly generation.

**Dataset:** Simple compiler only (105,261 functions, 54,844 FIM examples)  
**Model:** Salesforce/codegen-350M-mono  
**Training time:** ~2-3 hours on single GPU  
**Memory:** ~8GB VRAM  
**Expected accuracy:** 60-70% instruction accuracy

In [ ]:
# Install required packages
!pip install torch>=2.0.0
!pip install transformers>=4.35.0
!pip install datasets>=2.14.0
!pip install peft>=0.6.0
!pip install accelerate>=0.24.0

# Optional: Uncomment if you want monitoring
# !pip install wandb>=0.15.0
# !pip install tensorboard>=2.14.0

print("✓ All packages installed successfully!")

## 1. Setup and Imports

In [ ]:
# Apply LoRA
if USE_LORA:
    print("\nApplying LoRA...")
    print(f"  Rank: {LORA_R}")
    print(f"  Alpha: {LORA_ALPHA}")
    print(f"  Dropout: {LORA_DROPOUT}")
    
    # Auto-detect target modules based on model architecture
    # Different models have different module names
    model_name_lower = MODEL_NAME.lower()
    
    if "codegen" in model_name_lower:
        target_modules = ["qkv_proj", "out_proj"]  # CodeGen architecture
    elif "starcoder" in model_name_lower or "santacoder" in model_name_lower:
        target_modules = ["c_attn", "c_proj"]  # StarCoder (GPT-BigCode)
    elif "codellama" in model_name_lower or "llama" in model_name_lower:
        target_modules = ["q_proj", "v_proj", "k_proj", "o_proj"]  # Llama architecture
    elif "gpt2" in model_name_lower or "gpt-2" in model_name_lower:
        target_modules = ["c_attn", "c_proj"]  # GPT-2
    else:
        # Default: try to find common attention modules
        target_modules = ["q_proj", "v_proj"]  # Most common
        print(f"  Warning: Unknown model architecture, using default target_modules")
    
    print(f"  Target modules: {target_modules}")

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

## 1.1 Utility Functions

In [ ]:
def extract_zip(zip_path, extract_to='.'):
    """Extract zip file to specified directory.
    
    Args:
        zip_path: Path to zip file
        extract_to: Directory to extract to (default: current directory)
    
    Returns:
        Path to extracted directory
    """
    zip_path = Path(zip_path)
    extract_to = Path(extract_to)
    
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    
    print(f"Extracting {zip_path}...")
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    
    print(f"✓ Extracted to: {extract_to}")
    return extract_to

def create_model_archive(model_dir, archive_name=None):
    """Create zip archive of trained model.
    
    Args:
        model_dir: Path to model directory
        archive_name: Name of archive (default: model_dir.zip)
    
    Returns:
        Path to created archive
    """
    model_dir = Path(model_dir)
    
    if not model_dir.exists():
        raise FileNotFoundError(f"Model directory not found: {model_dir}")
    
    if archive_name is None:
        archive_name = f"{model_dir.name}.zip"
    
    archive_path = model_dir.parent / archive_name
    
    print(f"Creating archive: {archive_path}")
    
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in model_dir.rglob('*'):
            if file_path.is_file():
                arcname = file_path.relative_to(model_dir.parent)
                zipf.write(file_path, arcname)
                print(f"  Added: {arcname}")
    
    size_mb = archive_path.stat().st_size / (1024 * 1024)
    print(f"✓ Archive created: {archive_path} ({size_mb:.1f} MB)")
    
    return archive_path

print("Utility functions loaded")

## 2. Configuration

In [ ]:
# Model and dataset
MODEL_NAME = 'Salesforce/codegen-350M-mono'
DATASET_PATH = './data'  # Local data directory
DATASET_TYPE = 'fim'  # or 'plain'

# Training parameters
NUM_EPOCHS = 5
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
MAX_LENGTH = 512

# LoRA parameters
USE_LORA = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Output
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = f'./output/codegen-350M-simple-fim-{timestamp}'

# FIM tokens
FIM_PREFIX = "<fim_prefix>"
FIM_SUFFIX = "<fim_suffix>"
FIM_MIDDLE = "<fim_middle>"

print("Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Dataset: {DATASET_PATH}/{DATASET_TYPE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  LoRA: {USE_LORA}")
print(f"  Output: {OUTPUT_DIR}")

### ⚠️ Important: Dataset Setup

**Current Status:** The `data/` folder contains **SAMPLE DATA ONLY** (140 examples).

This is enough to test the notebook runs, but **NOT enough for real training**.

---

### Option 1: Use Sample Data (Testing Only)

The notebook will run with current data, but:
- Only 100 training examples
- Model won't learn properly
- Good for testing the pipeline works

**To use:** Just run the cells below as-is.

---

### Option 2: Use Full Dataset (Real Training)

**Full dataset location:** `../data/datasets/simple_optimal/`

**Statistics:**
- 105,261 functions
- 54,844 FIM examples (recommended)
- 105,261 plain examples

**Setup Methods:**

**A) Symlink (recommended - no disk space):**


In [ ]:
# OPTION A: Symlink to full dataset (recommended)
# Uncomment to use:
# !rm -rf data
# !ln -s ../data/datasets/simple_optimal data
# print("✓ Linked to full dataset")

# OPTION B: Copy full dataset (~60MB)
# Uncomment to use:
# !rm -rf data
# !cp -r ../data/datasets/simple_optimal data
# print("✓ Copied full dataset")

# Check current dataset size
import os
fim_train = 'data/fim/train.jsonl'
if os.path.exists(fim_train):
    lines = sum(1 for _ in open(fim_train))
    print(f"Current FIM training examples: {lines}")
    if lines < 1000:
        print("⚠️  Using SAMPLE data - not enough for real training!")
        print("   Uncomment one of the options above to use full dataset.")
    else:
        print("✓ Using FULL dataset")
else:
    print("❌ Data folder not found!")


## 3. Load Dataset

In [ ]:
print("Loading dataset...")

dataset = load_dataset(
    'json',
    data_files={
        'train': str(Path(DATASET_PATH) / DATASET_TYPE / 'train.jsonl'),
        'validation': str(Path(DATASET_PATH) / DATASET_TYPE / 'validation.jsonl'),
        'test': str(Path(DATASET_PATH) / DATASET_TYPE / 'test.jsonl')
    }
)

print(f"\nDataset loaded:")
print(f"  Train: {len(dataset['train'])} examples")
print(f"  Validation: {len(dataset['validation'])} examples")
print(f"  Test: {len(dataset['test'])} examples")

# Show example
print("\nExample:")
print(dataset['train'][0]['text'][:500])

## 4. Load Model and Tokenizer

In [ ]:
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Add FIM tokens
if DATASET_TYPE == 'fim':
    special_tokens = [FIM_PREFIX, FIM_SUFFIX, FIM_MIDDLE]
    tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer vocab size: {len(tokenizer)}")

In [ ]:
print("\nLoading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None
)

# Resize embeddings for FIM tokens
if DATASET_TYPE == 'fim':
    model.resize_token_embeddings(len(tokenizer))

print(f"Model loaded: {model.num_parameters():,} parameters")

In [ ]:
# Apply LoRA
if USE_LORA:
    print("\nApplying LoRA...")
    print(f"  Rank: {LORA_R}")
    print(f"  Alpha: {LORA_ALPHA}")
    print(f"  Dropout: {LORA_DROPOUT}")

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["c_attn", "c_proj", "c_fc"]
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

## 5. Preprocess Dataset

In [ ]:
def preprocess_function(examples):
    """Tokenize examples."""
    texts = examples['text']

    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors=None
    )

    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

print("Preprocessing dataset...")

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing"
)

print("Preprocessing complete")

## 6. Setup Training

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=100,
    logging_steps=50,
    eval_steps=500,
    save_steps=1000,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
    dataloader_num_workers=4,
    save_total_limit=3
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("Trainer ready")

## 7. Train Model

In [ ]:
print("\n" + "="*80)
print("Starting Training")
print("="*80)

trainer.train()

print("\nTraining complete!")

## 8. Save Model

In [ ]:
print("Saving model...")

final_model_path = Path(OUTPUT_DIR) / 'final_model'
trainer.save_model(str(final_model_path))
tokenizer.save_pretrained(str(final_model_path))

print(f"Model saved to: {final_model_path}")

## 9. Evaluate on Test Set

In [ ]:
print("\nEvaluating on test set...")

test_results = trainer.evaluate(tokenized_dataset['test'])

print("\nTest Results:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")

# Save results
results_path = Path(OUTPUT_DIR) / 'test_results.json'
with open(results_path, 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"\nResults saved to: {results_path}")

## 10. Test Generation

In [ ]:
def generate_completion(model, tokenizer, prompt, max_length=256):
    """Generate completion for prompt."""
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)

    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.8,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prompt_decoded = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)

    if generated.startswith(prompt_decoded):
        completion = generated[len(prompt_decoded):]
    else:
        completion = generated

    return completion.strip()

# Test with FIM example
if DATASET_TYPE == 'fim':
    prefix = """factorial:
    addi sp, sp, -32
    sw ra, 28(sp)
    sw s0, 24(sp)
    addi s0, sp, 32
    sw a0, -20(s0)
    lw a5, -20(s0)"""

    suffix = """    lw ra, 28(sp)
    lw s0, 24(sp)
    addi sp, sp, 32
    jr ra"""

    prompt = f"{FIM_PREFIX}{prefix}{FIM_SUFFIX}{suffix}{FIM_MIDDLE}"

    print("\n" + "="*80)
    print("Test Generation (FIM)")
    print("="*80)
    print("\nPrefix:")
    print(prefix)
    print("\nSuffix:")
    print(suffix)
    print("\nGenerated Middle:")
    print(generate_completion(model, tokenizer, prompt))

## 11. Package Model for Download

In [ ]:
print("\n" + "="*80)
print("Packaging Model for Download")
print("="*80)

# Create model archive
archive_name = f"riscv_codegen_350M_{timestamp}.zip"
archive_path = create_model_archive(final_model_path, archive_name)

print(f"\n✓ Model packaged and ready for download!")
print(f"\nArchive: {archive_path}")
print(f"Size: {archive_path.stat().st_size / (1024*1024):.1f} MB")

# Create summary file
summary = {
    'model_name': MODEL_NAME,
    'dataset': f'simple_optimal_{DATASET_TYPE}',
    'training_date': timestamp,
    'epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'lora_config': {
        'r': LORA_R,
        'alpha': LORA_ALPHA,
        'dropout': LORA_DROPOUT
    } if USE_LORA else None,
    'test_results': test_results,
    'archive_path': str(archive_path),
    'archive_size_mb': round(archive_path.stat().st_size / (1024*1024), 1)
}

summary_path = archive_path.parent / f"training_summary_{timestamp}.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nTraining summary: {summary_path}")
print("\n" + "="*80)
print("Training Complete!")
print("="*80)

## 12. Load Packaged Model (Optional)

Use this cell to test loading the model from the zip archive:

In [ ]:
# Example: Extract and load model from archive
# Uncomment to use:

# # Extract archive
# extracted_dir = extract_zip(archive_path, './extracted_models')
# 
# # Load model
# model_path = extracted_dir / final_model_path.name
# loaded_model = AutoModelForCausalLM.from_pretrained(model_path)
# loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
# 
# print(f"✓ Model loaded from archive: {model_path}")
# print(f"  Parameters: {loaded_model.num_parameters():,}")

## Summary

Training complete! Files created:

1. **Model directory:** `{OUTPUT_DIR}/final_model/`
2. **Model archive:** `riscv_codegen_350M_{timestamp}.zip` (ready for download)
3. **Training summary:** `training_summary_{timestamp}.json`
4. **Test results:** `{OUTPUT_DIR}/test_results.json`

**Next steps:**
1. Download the model archive
2. Run comprehensive evaluation using `evaluate_model.py`
3. Test interactive generation with `generate_assembly.py`
4. Compare with combined dataset results
5. Try larger models (StarCoder 3B, CodeLlama 7B)